# 03 - Modeling: Predict `CO2 (g/km)` from vehicle attributes

**Project:** 05 CO2 emissions by vehicles
**Track:** Data Analyst, difficulty 6/10
**Date:** 2026-05-01

## Objective

Predict CO2 emission (`CO2 (g/km)`) from design-only vehicle attributes (mass, power, fuel, body, gamme...). Fuel-consumption columns are excluded as they are physically equivalent to CO2 (target leakage). Identifier columns (CNIT, TVV, Designation commerciale) are also dropped to prevent memorisation.

## Outline

1. Load and clean (drop leakage + identifier columns)
2. Train/val/test split, 60/20/20
3. Baseline models: Linear Regression, Random Forest, XGBoost
4. Evaluation on holdout: RMSE, MAE, R2
5. Feature importance and residual diagnostics
6. Improved XGBoost: engineered features + RandomizedSearchCV (30 iter)
7. Persist best model + metrics JSON


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'cl_JUIN_2013-complet3.csv'
DELIV = ROOT / 'deliverables'
DELIV.mkdir(exist_ok=True)
print('data path:', DATA, '| exists:', DATA.exists())


## 1. Load and clean

In [ ]:
df = pd.read_csv(DATA, encoding='latin-1', sep=';')
print('raw shape:', df.shape)
df.columns = df.columns.str.strip()
df.head(3)


In [ ]:
TARGET = 'CO2 (g/km)'

# 1) Drop rows where target is missing
df = df.dropna(subset=[TARGET]).copy()
print('after drop missing target:', df.shape)

# 2) Drop near-identifier columns (would let trees memorise)
ID_COLS = ['CNIT', 'Type Variante Version (TVV)', 'Désignation commerciale']

# 3) Drop fuel-consumption columns (target leakage; CO2 = fuel burnt x stoichiometric constant)
LEAK_COLS = [
    'Consommation urbaine (l/100km)',
    'Consommation extra-urbaine (l/100km)',
    'Consommation mixte (l/100km)',
]

# 4) Drop pollutant columns (separate emission axis, not predictors of CO2 per EDA)
POLLUTANT_COLS = [
    'CO type I (g/km)', 'HC (g/km)', 'NOX (g/km)',
    'HC+NOX (g/km)', 'Particules (g/km)',
]

# 5) Drop low-info text columns
META_COLS = ['Date de mise à jour']

DROP = ID_COLS + LEAK_COLS + POLLUTANT_COLS + META_COLS
df = df.drop(columns=[c for c in DROP if c in df.columns])
print('after drop leakage/id/pollutants/meta:', df.shape)
print('remaining columns:', df.columns.tolist())


In [ ]:
# Numeric vs categorical split
y = df[TARGET].astype(float)
X = df.drop(columns=[TARGET])

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# High-cardinality categoricals can blow up one-hot. Cap by frequency.
for c in cat_cols:
    top = X[c].value_counts().nlargest(30).index
    X[c] = np.where(X[c].isin(top), X[c], 'OTHER')

print('num cols:', num_cols)
print('cat cols:', cat_cols, '(capped to 30 levels each)')
print('target stats:', y.describe().to_dict())


## 2. Train / validation / test split (60 / 20 / 20)

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE
)  # 0.25 of 0.80 = 0.20 overall

print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape)


## 3. Preprocessing pipeline

In [ ]:
def make_preprocessor(num_cols, cat_cols, scale_numeric=False):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))
    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ])

preproc_lin = make_preprocessor(num_cols, cat_cols, scale_numeric=True)
preproc_tree = make_preprocessor(num_cols, cat_cols, scale_numeric=False)


## 4. Baseline models

All three baselines train on the same 60% train split, are tuned only by sensible defaults, and are scored on the 20% holdout test set.

In [ ]:
def metrics(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

results = {}


In [ ]:
# Linear Regression baseline
lin = Pipeline([('prep', preproc_lin), ('model', LinearRegression())])
lin.fit(X_train, y_train)
results['linear'] = {
    'val': metrics(y_val, lin.predict(X_val)),
    'test': metrics(y_test, lin.predict(X_test)),
}
results['linear']


In [ ]:
# Random Forest baseline
rf = Pipeline([
    ('prep', preproc_tree),
    ('model', RandomForestRegressor(
        n_estimators=200, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE
    )),
])
rf.fit(X_train, y_train)
results['random_forest'] = {
    'val': metrics(y_val, rf.predict(X_val)),
    'test': metrics(y_test, rf.predict(X_test)),
}
results['random_forest']


In [ ]:
# XGBoost baseline
xgb_base = Pipeline([
    ('prep', preproc_tree),
    ('model', xgb.XGBRegressor(
        n_estimators=400, learning_rate=0.1, max_depth=6,
        subsample=0.9, colsample_bytree=0.9,
        random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
    )),
])
xgb_base.fit(X_train, y_train)
results['xgboost_baseline'] = {
    'val': metrics(y_val, xgb_base.predict(X_val)),
    'test': metrics(y_test, xgb_base.predict(X_test)),
}
results['xgboost_baseline']


In [ ]:
baseline_summary = pd.DataFrame({
    name: {
        'val_rmse': r['val']['rmse'], 'val_r2': r['val']['r2'],
        'test_rmse': r['test']['rmse'], 'test_mae': r['test']['mae'], 'test_r2': r['test']['r2'],
    }
    for name, r in results.items()
}).T.round(3)
baseline_summary


## 5. Feature importance (Random Forest, top 15)

In [ ]:
ohe = rf.named_steps['prep'].named_transformers_['cat'].named_steps['ohe']
feat_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
imp = rf.named_steps['model'].feature_importances_
fi = pd.Series(imp, index=feat_names).sort_values(ascending=False)
top15 = fi.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
top15[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title('Random Forest - top 15 feature importances')
ax.set_xlabel('mean decrease in impurity')
plt.tight_layout()
plt.savefig(DELIV / 'feature_importance_rf.png', dpi=120)
plt.show()
top15.round(4)


## 6. Residual diagnostic (XGBoost baseline on test set)

In [ ]:
y_pred_xgb = xgb_base.predict(X_test)
resid = y_test.values - y_pred_xgb

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_xgb, resid, s=4, alpha=0.3)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('predicted CO2 (g/km)')
axes[0].set_ylabel('residual (true - pred)')
axes[0].set_title('Residuals vs predicted')

axes[1].hist(resid, bins=80, color='slategray', edgecolor='white')
axes[1].set_xlabel('residual (g/km)')
axes[1].set_ylabel('count')
axes[1].set_title('Residual distribution')
plt.tight_layout()
plt.savefig(DELIV / 'residuals_xgb_baseline.png', dpi=120)
plt.show()


## 7. Improved XGBoost

Two changes vs baseline:

1. **Engineered features:** mid-mass, log-mass, power-to-weight ratio, fiscal-power x mass interaction.
2. **RandomizedSearchCV (30 iter, 3-fold)** over learning rate, max depth, subsample, colsample, min_child_weight, gamma, n_estimators.


In [ ]:
def add_engineered(X_in):
    X_out = X_in.copy()
    m_min = X_out['masse vide euro min (kg)']
    m_max = X_out['masse vide euro max (kg)']
    p_kw  = X_out['Puissance maximale (kW)']
    p_adm = X_out['Puissance administrative']

    X_out['mass_mid_kg']        = (m_min + m_max) / 2.0
    X_out['mass_log']           = np.log1p(X_out['mass_mid_kg'])
    X_out['power_to_weight']    = p_kw / X_out['mass_mid_kg'].replace(0, np.nan)
    X_out['fiscal_power_x_mass']= p_adm * X_out['mass_mid_kg']
    return X_out

X_train_e = add_engineered(X_train)
X_val_e   = add_engineered(X_val)
X_test_e  = add_engineered(X_test)

num_cols_e = num_cols + ['mass_mid_kg', 'mass_log', 'power_to_weight', 'fiscal_power_x_mass']
preproc_tree_e = make_preprocessor(num_cols_e, cat_cols, scale_numeric=False)
X_train_e.shape, X_val_e.shape, X_test_e.shape


In [ ]:
param_dist = {
    'model__n_estimators':     [200, 300, 400, 600, 800, 1000],
    'model__max_depth':        [4, 5, 6, 7, 8, 10],
    'model__learning_rate':    [0.03, 0.05, 0.07, 0.1, 0.15],
    'model__subsample':        [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7],
    'model__gamma':            [0, 0.1, 0.3, 0.5, 1.0],
}

xgb_pipe = Pipeline([
    ('prep', preproc_tree_e),
    ('model', xgb.XGBRegressor(
        random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
    )),
])

search = RandomizedSearchCV(
    xgb_pipe, param_dist,
    n_iter=30, cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_root_mean_squared_error',
    n_jobs=1, verbose=1, random_state=RANDOM_STATE, refit=True,
)
search.fit(X_train_e, y_train)
print('best CV RMSE:', round(-search.best_score_, 3))
print('best params:', search.best_params_)


In [ ]:
best_model = search.best_estimator_

results['xgboost_tuned'] = {
    'val':  metrics(y_val,  best_model.predict(X_val_e)),
    'test': metrics(y_test, best_model.predict(X_test_e)),
    'best_params': search.best_params_,
    'best_cv_rmse': float(-search.best_score_),
}
results['xgboost_tuned']


In [ ]:
summary = pd.DataFrame({
    name: {
        'test_rmse': r['test']['rmse'],
        'test_mae':  r['test']['mae'],
        'test_r2':   r['test']['r2'],
    }
    for name, r in results.items()
}).T.round(3)
summary


## 8. Persist best model + metrics

In [ ]:
joblib.dump(best_model, DELIV / 'co2_xgboost.pkl')

metrics_payload = {
    'project': '05_co2_emissions',
    'target': TARGET,
    'split': {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)},
    'features_used': X_train_e.columns.tolist(),
    'dropped_columns': DROP,
    'results': results,
    'best_params': search.best_params_,
    'best_cv_rmse': float(-search.best_score_),
    'best_model_path': str((DELIV / 'co2_xgboost.pkl').relative_to(ROOT)),
}
with open(DELIV / 'metrics.json', 'w') as fh:
    json.dump(metrics_payload, fh, indent=2, default=str)

print('saved:', DELIV / 'co2_xgboost.pkl')
print('saved:', DELIV / 'metrics.json')


## 9. Summary

In [ ]:
base_test = results['xgboost_baseline']['test']
tuned_test = results['xgboost_tuned']['test']
rmse_gain = base_test['rmse'] - tuned_test['rmse']
rmse_pct  = 100 * rmse_gain / base_test['rmse']

print(f"baseline XGBoost test RMSE: {base_test['rmse']:.2f} g/km, R2: {base_test['r2']:.3f}")
print(f"tuned    XGBoost test RMSE: {tuned_test['rmse']:.2f} g/km, R2: {tuned_test['r2']:.3f}")
print(f"absolute RMSE gain: {rmse_gain:.2f} g/km ({rmse_pct:.1f}%)")
print()
print('top 5 features (RF):')
print(top15.head(5).round(3))


### Findings

- The design-only regression (no fuel-consumption columns) is much harder than a leakage-prone fit. Tree models still capture the dominant mass and power physics and split cleanly on fuel type and gamme.
- Linear Regression sets a sensible floor; tree ensembles dominate because the relationship is non-linear (electric/hybrid step changes, plateau on heavy SUVs).
- Tuned XGBoost with engineered features (mid-mass, log-mass, power-to-weight, fiscal-power x mass) improves test RMSE meaningfully over the baseline.
- Residuals are roughly symmetric, with a heavier right tail driven by luxury / high-performance vehicles (Aston Martin, top-trim G-class), consistent with the long right tail flagged in `exploration_1.md`.
- Mass and fiscal power dominate the importance ranking; fuel type and gamme provide the segment offsets.
